# Curriculum experiment — Round 2: does ordering matter when fine-tuning actually helps?

**Round 1 result:** fine-tuning at LR 2e-4 with full-sequence loss *damaged* Qwen2.5-0.5B-Instruct (baseline 43.0% → A 34.0%, B 30.5%, C 32.0%). Ordering couldn't be judged in a harmful-training regime.

**Round 2 changes (only two, deliberately):**
1. **Learning rate 2e-5** (10x gentler, still constant for fairness between orderings)
2. **Prompt masking** — loss is computed only on the answer tokens, so the model isn't trained to regenerate questions

Everything else is identical to round 1: same model, same 2,000 problems, same three orderings (A shuffled / B easy→hard / C hard→easy), same 200 held-out test problems.

**Success criterion for the setup:** run A must beat the 43.0% baseline. Only then can B vs A vs C be interpreted.

**Prediction on record (Claude):** all three beat baseline; orderings within ~2% of each other.

Setup: `Runtime → Change runtime type → T4 GPU`, then Run all. ~1.5–2 hours.

In [1]:
# Cell 1: Install (includes the torchao fix from round 1)
!pip uninstall -y -q torchao
!pip install -q -U transformers datasets peft accelerate

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.3 MB/s eta 0:00:00
GPU: Tesla T4


In [2]:
# Cell 2: Data — identical selection and orderings to round 1
import random
from datasets import load_dataset

SEED = 42
N_TRAIN = 2000
N_EVAL = 200

gsm = load_dataset("openai/gsm8k", "main")

def difficulty(example):
    return example["answer"].count("<<")

train_pool = list(gsm["train"])
random.Random(SEED).shuffle(train_pool)
train_pool = train_pool[:N_TRAIN]
eval_pool = list(gsm["test"])[:N_EVAL]

order_A = list(train_pool)
random.Random(SEED + 1).shuffle(order_A)
order_B = sorted(train_pool, key=difficulty)
order_C = sorted(train_pool, key=difficulty, reverse=True)

print(f"Train: {len(train_pool)}, eval: {len(eval_pool)} — same data as round 1")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train: 2000, eval: 200 — same data as round 1


In [3]:
# Cell 3: Tokenization WITH PROMPT MASKING (round-2 change #2)
# Labels are -100 for prompt tokens -> loss computed only on the answer.
import re
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LEN = 640

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

PROMPT_PREFIX = "Solve this math problem step by step. End with the final answer after '####'.\n\n"

def tokenize_masked(ex):
    answer = re.sub(r"<<[^>]*>>", "", ex["answer"])
    user_msg = [{"role": "user", "content": PROMPT_PREFIX + ex["question"]}]
    full_msg = user_msg + [{"role": "assistant", "content": answer}]

    # Tokenize prompt-only (with generation prompt) and full conversation
    prompt_text = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_msg, tokenize=False)

    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_LEN)["input_ids"]
    full = tokenizer(full_text, truncation=True, max_length=MAX_LEN)

    labels = list(full["input_ids"])
    n_prompt = min(len(prompt_ids), len(labels))
    labels[:n_prompt] = [-100] * n_prompt          # mask the prompt

    return {"input_ids": full["input_ids"],
            "attention_mask": full["attention_mask"],
            "labels": labels}

dataset_A = [tokenize_masked(e) for e in order_A]
dataset_B = [tokenize_masked(e) for e in order_B]
dataset_C = [tokenize_masked(e) for e in order_C]

# Sanity check the masking on one example
ex0 = dataset_A[0]
n_masked = sum(1 for l in ex0["labels"] if l == -100)
print(f"Example 0: {len(ex0['input_ids'])} tokens, {n_masked} masked (prompt), {len(ex0['input_ids'])-n_masked} trained (answer)")
assert 0 < n_masked < len(ex0["labels"]), "Masking looks wrong!"
print("✓ Prompt masking active")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Example 0: 159 tokens, 97 masked (prompt), 62 trained (answer)
✓ Prompt masking active


In [4]:
# Cell 4: Training — LR 2e-5 (round-2 change #1), custom collator pads labels with -100
import gc
import torch.nn as nn
from torch.utils.data import Dataset, SequentialSampler
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

class ListDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

class OrderedTrainer(Trainer):
    """HF Trainer shuffles by default — this forces sequential order."""
    def _get_train_sampler(self, *args, **kwargs):
        return SequentialSampler(self.train_dataset)

def collate_masked(batch):
    """Right-pad input_ids/attention_mask with pad token, labels with -100."""
    max_len = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, attn, labels = [], [], []
    for x in batch:
        n = max_len - len(x["input_ids"])
        input_ids.append(list(x["input_ids"]) + [pad_id] * n)
        attn.append(list(x["attention_mask"]) + [0] * n)
        labels.append(list(x["labels"]) + [-100] * n)
    return {"input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn),
            "labels": torch.tensor(labels)}

def train_one_run(run_name, dataset):
    print(f"\n{'='*50}\nTraining run {run_name}\n{'='*50}")
    torch.manual_seed(SEED)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model.config.use_cache = False

    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, lora)

    args = TrainingArguments(
        output_dir=f"./out_{run_name}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,             # ROUND-2 CHANGE: 10x gentler than round 1
        lr_scheduler_type="constant",
        warmup_steps=10,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        report_to="none",
        seed=SEED,
    )

    trainer = OrderedTrainer(model=model, args=args,
                             train_dataset=ListDataset(dataset),
                             data_collator=collate_masked)
    trainer.train()

    model.save_pretrained(f"./adapter_r2_{run_name}")
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    print(f"Run {run_name} done — saved to ./adapter_r2_{run_name}")

In [5]:
# Cell 5: Sanity check — order preserved? (fixed dummy-model version from round 1)
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, **kwargs):
        return None

test_trainer = OrderedTrainer(
    model=DummyModel(),
    args=TrainingArguments(output_dir="./tmp", report_to="none"),
    train_dataset=ListDataset(list(range(100))),
)
try:
    sampler = test_trainer._get_train_sampler()
except TypeError:
    sampler = test_trainer._get_train_sampler(test_trainer.train_dataset)
order = list(sampler)
assert order == list(range(100)), f"ORDER BROKEN! First 10: {order[:10]}"
print("✓ Order preserved — experiment is valid")

✓ Order preserved — experiment is valid


In [6]:
# Cell 6: Run all three trainings (resume-friendly)
import os
for name, ds in [("A_shuffled", dataset_A), ("B_easy_to_hard", dataset_B), ("C_hard_to_easy", dataset_C)]:
    if os.path.exists(f"./adapter_r2_{name}"):
        print(f"Skipping {name} — adapter exists")
    else:
        train_one_run(name, ds)


Training run A_shuffled


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Step,Training Loss
50,0.675997
100,0.586470
150,0.529633
200,0.517902
250,0.512696


Run B_easy_to_hard done — saved to ./adapter_r2_B_easy_to_hard

Training run C_hard_to_easy


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
50,0.567940
100,0.538531
150,0.528984
200,0.549229
250,0.598941


Run C_hard_to_easy done — saved to ./adapter_r2_C_hard_to_easy


In [7]:
# Cell 7: Evaluation — identical to round 1
from peft import PeftModel

def extract_gold(answer_text):
    return answer_text.split("####")[-1].strip().replace(",", "")

def extract_pred(generated_text):
    if "####" in generated_text:
        tail = generated_text.split("####")[-1]
        nums = re.findall(r"-?\d+\.?\d*", tail.replace(",", ""))
        if nums: return nums[0].rstrip(".")
    nums = re.findall(r"-?\d+\.?\d*", generated_text.replace(",", ""))
    return nums[-1].rstrip(".") if nums else None

def norm(x):
    try: return float(x)
    except (TypeError, ValueError): return None

@torch.no_grad()
def evaluate(adapter_path, label, batch_size=8):
    print(f"\nEvaluating {label} ...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    model.eval()

    tokenizer.padding_side = "left"
    correct = 0
    for i in range(0, len(eval_pool), batch_size):
        batch = eval_pool[i:i+batch_size]
        prompts = [tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT_PREFIX + ex["question"]}],
            tokenize=False, add_generation_prompt=True) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to("cuda")
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j, ex in enumerate(batch):
            gen = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            if norm(extract_pred(gen)) is not None and norm(extract_pred(gen)) == norm(extract_gold(ex["answer"])):
                correct += 1
        if (i // batch_size) % 5 == 0:
            print(f"  {i+len(batch)}/{len(eval_pool)} — running accuracy: {correct/(i+len(batch)):.1%}")

    del model, base
    gc.collect(); torch.cuda.empty_cache()
    acc = correct / len(eval_pool)
    print(f"{label}: {correct}/{len(eval_pool)} = {acc:.1%}")
    return acc

results = {}
results["baseline (no fine-tune)"] = evaluate(None, "baseline (no fine-tune)")
results["A: shuffled"]             = evaluate("./adapter_r2_A_shuffled", "A: shuffled")
results["B: easy \u2192 hard"]       = evaluate("./adapter_r2_B_easy_to_hard", "B: easy → hard")
results["C: hard \u2192 easy"]       = evaluate("./adapter_r2_C_hard_to_easy", "C: hard → easy")


Evaluating baseline (no fine-tune) ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 37.5%
  48/200 — running accuracy: 31.2%
  88/200 — running accuracy: 35.2%
  128/200 — running accuracy: 39.1%
  168/200 — running accuracy: 41.1%
baseline (no fine-tune): 86/200 = 43.0%

Evaluating A: shuffled ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 25.0%
  48/200 — running accuracy: 25.0%
  88/200 — running accuracy: 31.8%
  128/200 — running accuracy: 33.6%
  168/200 — running accuracy: 36.9%
A: shuffled: 72/200 = 36.0%

Evaluating B: easy → hard ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 25.0%
  48/200 — running accuracy: 20.8%
  88/200 — running accuracy: 26.1%
  128/200 — running accuracy: 31.2%
  168/200 — running accuracy: 33.9%
B: easy → hard: 67/200 = 33.5%

Evaluating C: hard → easy ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  8/200 — running accuracy: 12.5%
  48/200 — running accuracy: 16.7%
  88/200 — running accuracy: 22.7%
  128/200 — running accuracy: 25.8%
  168/200 — running accuracy: 28.6%
C: hard → easy: 58/200 = 29.0%


In [8]:
# Cell 8: Final results, both rounds side by side
round1 = {"baseline (no fine-tune)": 0.430, "A: shuffled": 0.340,
          "B: easy \u2192 hard": 0.305, "C: hard \u2192 easy": 0.320}

print("\n" + "="*58)
print("RESULTS — GSM8K accuracy, 200 held-out problems")
print("="*58)
print(f"{'':<28}{'Round 1':>12}{'Round 2':>12}")
print(f"{'':<28}{'(LR 2e-4)':>12}{'(LR 2e-5,':>12}")
print(f"{'':<28}{'':>12}{'masked)':>12}")
print("-"*58)
for k in round1:
    r2 = results.get(k)
    print(f"{k:<28}{round1[k]:>11.1%}{r2:>12.1%}")
print("="*58)
print("\nInterpretation guide:")
print("1. A > baseline?  If NO, the setup still isn't in a helpful regime")
print("   and orderings can't be judged. If YES, continue:")
print("2. B > A by 5%+ and C <= A  -> evidence FOR curriculum")
print("3. All within ~2%           -> ordering doesn't matter at this scale")
print("\nPrediction on record (Claude): all three beat baseline,")
print("orderings within ~2% of each other.")


RESULTS — GSM8K accuracy, 200 held-out problems
                                 Round 1     Round 2
                               (LR 2e-4)   (LR 2e-5,
                                             masked)
----------------------------------------------------------
baseline (no fine-tune)           43.0%       43.0%
A: shuffled                       34.0%       36.0%
B: easy → hard                    30.5%       33.5%
C: hard → easy                    32.0%       29.0%

Interpretation guide:
1. A > baseline?  If NO, the setup still isn't in a helpful regime
   and orderings can't be judged. If YES, continue:
2. B > A by 5%+ and C <= A  -> evidence FOR curriculum
3. All within ~2%           -> ordering doesn't matter at this scale

Prediction on record (Claude): all three beat baseline,
orderings within ~2% of each other.
